In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.layers import MaxPooling1D, Flatten, Dense
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
import zipfile
from tensorflow.keras.layers import Input, Conv1D
# from catboost import CatBoostClassifier
from google.colab import files
import lightgbm as lgb
import os
# !pip install catboost
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from keras.models import Sequential
from tensorflow.keras import layers
from sklearn.model_selection import RandomizedSearchCV
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Activation, BatchNormalization
from keras.layers import Input
# السطر الجديد اللي هيحل الأزمة:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import re
from scipy.interpolate import interp1d
# أولاً: تسطيب المكتبة الخاصة ببيانات ناسا
!pip install lightkurve --quiet
import matplotlib.pyplot as plt
import lightkurve as lk

In [ ]:
# 1. القوائم الذهبية للـ 60 نجم
planets = [
    "Kepler-1", "Kepler-2", "Kepler-4", "Kepler-5", "Kepler-6",
    "Kepler-7", "Kepler-8", "Kepler-10", "Kepler-11", "Kepler-12",
    "Kepler-13", "Kepler-14", "Kepler-15", "Kepler-17", "Kepler-18",
    "Kepler-20", "Kepler-22", "Kepler-40", "Kepler-62", "Kepler-80",
    "Kepler-90", "Kepler-186", "Kepler-452", "Kepler-1649", "Kepler-296"
]
no_planets = [
    "KIC 1026473", "KIC 1725848", "KIC 3247294", "KIC 3832716","KIC 6678383","KIC 8292840","KIC 10001893","KIC 11242721",
    "KIC 8462852", "KIC 5522786", "KIC 9832227", "KIC 10227020", "KIC 11442793",
    "KIC 3427344", "KIC 2157356", "KIC 10187017", "KIC 10001893", "KIC 5807616",
    "KIC 11026764", "KIC 3542116", "KIC 2436324", "KIC 4544587", "KIC 5613330",
    "KIC 6442183", "KIC 7289100", "KIC 8106973", "KIC 9025370", "KIC 11853905"
]       #(ملاحظة: النجوم دي بنستخدم لها اسم الكتالوج KIC وهو اختصار Kepler Input Catalog عشان التلسكوب يعرفهم).
binaries = [
    "KIC 10001167", "KIC 10020466", "KIC 10022411", "KIC 10156064", "KIC 10212484",
    "KIC 10260404", "KIC 10341828", "KIC 10417433", "KIC 10483507", "KIC 10544976","KIC 10226388","KIC 10544976","KIC 10910035","KIC 9592855"
]


all_stars = [(name, 1) for name in planets] + [(name, 0) for name in no_planets] + [(name, 2) for name in binaries]

data_features = []
labels = []

print("🚀 Starting the Grand 60-Star Mission...")

for name, label in all_stars:
    try:
        search = lk.search_lightcurve(name, author="Kepler", cadence="long")
        if len(search) == 0: continue # لو ملقاش بيانات كبلر يكمل
        lc = search[0].download()

        # التنظيف
        lc_clean = lc.remove_outliers().flatten()

        # البحث عن النبض (BLS)
        pg = lc_clean.to_periodogram(method='bls')
        best_p = pg.period_at_max_power
        t0_p = pg.transit_time_at_max_power

        lc_folded = lc_clean.fold(period=best_p, epoch_time=t0_p)
        bin_lc = lc_folded.bin(time_bin_size=best_p/500)

        flux_values = bin_lc.flux.value[:500] # نضمن إنهم 500 بالظبط

        if len(flux_values) == 500:
            # التعامل مع الـ NaN
            if np.isnan(flux_values).any():
                flux_values = np.where(np.isnan(flux_values), np.nanmean(flux_values), flux_values)

            data_features.append(flux_values)
            labels.append(label) # هنا بنحط 0 أو 1 أو 2 حسب القائمة
            print(f"✅ {name} Done (Label: {label})")

    except:
        print(f"❌ Skipping {name}")

X_raw = np.array(data_features)
y = np.array(labels)

X_min = X_raw.min()
X_max = X_raw.max()
X_scaled = (X_raw - X_min) / (X_max - X_min)
# 3. دلوقتي نوزعها لـ Global و Local زي ما اتعلمنا
X_global = X_scaled.copy() # الـ 500 نقطة كاملة
X_local = []

🚀 Starting the Grand 60-Star Mission...
✅ Kepler-4 Done (Label: 1)
✅ Kepler-5 Done (Label: 1)
✅ Kepler-8 Done (Label: 1)
✅ Kepler-11 Done (Label: 1)
✅ Kepler-12 Done (Label: 1)
✅ Kepler-13 Done (Label: 1)
✅ Kepler-15 Done (Label: 1)
✅ Kepler-17 Done (Label: 1)
✅ Kepler-18 Done (Label: 1)
✅ Kepler-20 Done (Label: 1)
✅ Kepler-40 Done (Label: 1)
✅ Kepler-62 Done (Label: 1)
✅ Kepler-80 Done (Label: 1)
✅ Kepler-90 Done (Label: 1)
✅ Kepler-186 Done (Label: 1)
✅ Kepler-1649 Done (Label: 1)
✅ Kepler-296 Done (Label: 1)
✅ KIC 1026473 Done (Label: 0)
✅ KIC 1725848 Done (Label: 0)
✅ KIC 3247294 Done (Label: 0)
✅ KIC 3832716 Done (Label: 0)
✅ KIC 6678383 Done (Label: 0)
✅ KIC 8292840 Done (Label: 0)
✅ KIC 10001893 Done (Label: 0)
✅ KIC 11242721 Done (Label: 0)
✅ KIC 8462852 Done (Label: 0)
✅ KIC 5522786 Done (Label: 0)
✅ KIC 10227020 Done (Label: 0)
✅ KIC 11442793 Done (Label: 0)


Could not resolve "KIC 3427344" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 3427344" to a sky position.


✅ KIC 2157356 Done (Label: 0)
✅ KIC 10001893 Done (Label: 0)
✅ KIC 5807616 Done (Label: 0)
✅ KIC 2436324 Done (Label: 0)
✅ KIC 4544587 Done (Label: 0)


Could not resolve "KIC 7289100" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 7289100" to a sky position.


✅ KIC 8106973 Done (Label: 0)
✅ KIC 11853905 Done (Label: 0)
✅ KIC 10156064 Done (Label: 2)
✅ KIC 10544976 Done (Label: 2)
✅ KIC 10226388 Done (Label: 2)
✅ KIC 10544976 Done (Label: 2)
✅ KIC 10910035 Done (Label: 2)


In [ ]:
def get_local_view(flux, period, t0, duration=0.2):
    center = len(flux) // 2
    local_view = flux[center-50 : center+50]
    return local_view

input_global = layers.Input(shape=(500, 1))
x1 = layers.Conv1D(16, 5, activation='relu')(input_global)
x1 = layers.MaxPooling1D(2)(x1)
x1 = layers.Flatten()(x1)

input_local = layers.Input(shape=(100, 1))
x2 = layers.Conv1D(16, 5, activation='relu')(input_local)
x2 = layers.MaxPooling1D(2)(x2)
x2 = layers.Flatten()(x2)

combined = layers.Concatenate()([x1, x2])
z = layers.Dense(32, activation='relu')(combined)
output=layers.Dense(3, activation='softmax')(z)
multi_model = models.Model(inputs=[input_global, input_local], outputs=output)
multi_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
multi_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 500, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 496, 16)   │         96 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 96, 16)    │         96 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 248, 16)   │          0 │ conv1d_2[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 48, 16)    │          0 │ conv1d_3[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 3968)      │          0 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 768)       │          0 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 4736)      │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │    151,584 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │         99 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 151,875 (593.26 KB)

 Trainable params: 151,875 (593.26 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
for flux in X_global:
    local = get_local_view(flux, None, None) # بنستخدم الفانكشن اللي شرحناها
    X_local.append(local)

X_local = np.array(X_local)
y = np.array(labels) # الإجابات (1 كوكب، 0 مش كوكب)

# تظبيط الأبعاد عشان الـ CNN (لازم نزيد بُعد "القناة")
X_global = X_global.reshape(-1, 500, 1)
X_local = X_local.reshape(-1, 100, 1)

print(f"Global shape: {X_global.shape}") # (عدد النجوم, 500, 1)
print(f"Local shape: {X_local.shape}")   # (عدد النجوم, 100, 1)
# تدريب الموديل المزدوج
history = multi_model.fit(
    [X_global, X_local], # المدخلات المزدوجة
    y,
    epochs=20,           # هيلف على الداتا 20 مرة
    batch_size=2,        # هياخد نجمين بنجمين
    verbose=1
)

In [ ]:
# عايزة تخزني بيانات بس؟ استعملي List.
# عايزة تعملي معادلة (زي الـ Scaling أو معادلات ميكانيكا الكم)؟ لازم تحوليها لـ Numpy Array.
# 1. قائمة النجوم اللي عايزين نكشف سرها
# --- 25 نجم يمتلكون كواكب مؤكدة (Label: 1) ---
new_planets = [
    "Kepler-3", "Kepler-9", "Kepler-16", "Kepler-21", "Kepler-25",
    "Kepler-28", "Kepler-32", "Kepler-35", "Kepler-37", "Kepler-42",
    "Kepler-45", "Kepler-48", "Kepler-51", "Kepler-54", "Kepler-57",
    "Kepler-60", "Kepler-65", "Kepler-68", "Kepler-70", "Kepler-75",
    "Kepler-77", "Kepler-82", "Kepler-84", "Kepler-86", "Kepler-89"
]

# --- 20 نجم "عادي" لا يوجد كواكب تعبر من أمامه (Label: 0) ---
# ملحوظة: دي نجوم ساطعة جداً بنستخدمها كمرجع للهدوء
new_no_planets = [
    "KIC 5000000", "KIC 5100000", "KIC 5200000", "KIC 5300000", "KIC 5400000",
    "KIC 5500000", "KIC 5600000", "KIC 5700000", "KIC 5800000", "KIC 5900000",
    "KIC 6000000", "KIC 6100000", "KIC 6200000", "KIC 6300000", "KIC 6400000",
    "KIC 6500000", "KIC 6600000", "KIC 6700000", "KIC 6800000", "KIC 6900000"
]

# --- 15 نجم ثنائي (نجمين بيلفوا حول بعض) (Label: 2) ---
# دي "الخدع" اللي الـ AI لازم يتعلم يتجنبها
new_binaries = [
    "KIC 11403044", "KIC 11446443", "KIC 11496155", "KIC 11506938", "KIC 11551430",
    "KIC 11558725", "KIC 11560447", "KIC 11569613", "KIC 11603133", "KIC 11613454",
    "KIC 11619962", "KIC 11621456", "KIC 11623456", "KIC 11656789", "KIC 11700000"
]

# دمج الجيش الجديد
stars_to_test = [(name, 1) for name in new_planets] + \
                [(name, 0) for name in new_no_planets] + \
                [(name, 2) for name in new_binaries]

for name_tuple in stars_to_test: # 1. التعديل هنا
    try:
        name = name_tuple[0]
        # --- خطوة 1: جلب البيانات وتنظيفها ---
        lc = lk.search_lightcurve(name, author="Kepler", cadence="long")[0].download()
        lc_clean = lc.remove_outliers().flatten()

        # --- خطوة 2: البحث عن النبض (الدورة الزمنية) ---
        pg = lc_clean.to_periodogram(method='bls')
        best_p = pg.period_at_max_power
        t0_p = pg.transit_time_at_max_power

        # --- خطوة 3: تجهيز الـ Global View (500 نقطة) ---
        lc_folded = lc_clean.fold(period=best_p, epoch_time=t0_p)
        bin_global = lc_folded.bin(time_bin_size=best_p/500)
        global_flux = bin_global.flux.value[:500]

        # التأكد من اكتمال البيانات (500 نقطة) والتعامل مع الـ NaN
        if len(global_flux) < 500:
            # تكملة النواقص بمتوسط القيم لو وجدت
            global_flux = np.pad(global_flux, (0, 500 - len(global_flux)), mode='mean')

        global_flux = np.where(np.isnan(global_flux), np.nanmean(global_flux), global_flux)

        # --- خطوة 4: تجهيز الـ Local View (100 نقطة زووم) ---
        local_flux = get_local_view(global_flux, None, None)

        # --- خطوة 5: الـ Scaling (توحيد المقاييس) ---
        # بنخلي القيم بين 0 و 1 عشان الـ AI ميتلخبطش بقوة إضاءة النجم
        g_min, g_max = np.min(global_flux), np.max(global_flux)
        global_scaled = (global_flux - g_min) / (g_max - g_min)

        l_min, l_max = np.min(local_flux), np.max(local_flux)
        local_scaled = (local_flux - l_min) / (l_max - l_min)

        # --- خطوة 6: تظبيط الأبعاد (Reshape) ---
        input_g = global_scaled.reshape(1, 500, 1)
        input_l = local_scaled.reshape(1, 100, 1)

        # --- خطوة 7: اللحظة الحاسمة (التنبؤ الثلاثي) ---
        result = multi_model.predict([input_g, input_l])

        # اختيار الفئة صاحبة أعلى احتمالية
        predicted_class = np.argmax(result[0])
        confidence = result[0][predicted_class] * 100

        # قاموس الفئات
        classes = {
            0: "Empty (Normal Star) 🌌🌌",
            1: "PLANET DETECTED! 🌟",
            2: "Binary Star (False Positive) ⚠️"
        }

        print(f"\n" + "="*30)
        print(f"🔭 Target: {name}")
        print(f"🧠 AI Classification: {classes[predicted_class]}")
        print(f"📊 Confidence: {confidence:.2f}%")
        print(f"⏱️ Orbital Period Found: {best_p:.2f} days")
        print("="*30)

    except Exception as e:
        print(f"❌ Error analysis for {name}: {e}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 860ms/step

🔭 Target: Kepler-3
🧠 AI Classification: Empty (Normal Star) 🌌🌌
📊 Confidence: nan%
⏱️ Orbital Period Found: 1.77 d days
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step

🔭 Target: Kepler-9
🧠 AI Classification: Empty (Normal Star) 🌌🌌
📊 Confidence: nan%
⏱️ Orbital Period Found: 9.62 d days
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step

🔭 Target: Kepler-16
🧠 AI Classification: Empty (Normal Star) 🌌🌌
📊 Confidence: nan%
⏱️ Orbital Period Found: 0.36 d days
❌ Error analysis for Kepler-21: no implementation found for 'numpy.pad' on types that implement __array_function__: [<class 'astropy.utils.masked.core.MaskedNDArray'>]
❌ Error analysis for Kepler-25: no implementation found for 'numpy.pad' on types that implement __array_function__: [<class 'astropy.utils.masked.core.MaskedNDArray'>]
❌ Error analysis for Kepler-28: no implementation found for 'numpy.pad' on types that implement __array_function__: [<class 'astropy.utils.masked.core.MaskedNDArray'>]
1/1 ━━━━━━━━━━━━━━

Could not resolve "KIC 5000000" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 5000000" to a sky position.


❌ Error analysis for KIC 5000000: index 0 out of range for table with length 0


Could not resolve "KIC 5100000" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 5100000" to a sky position.


❌ Error analysis for KIC 5100000: index 0 out of range for table with length 0


Could not resolve "KIC 5200000" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 5200000" to a sky position.


... (3 outputs left out)

Could not resolve "KIC 5400000" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 5400000" to a sky position.


❌ Error analysis for KIC 5400000: index 0 out of range for table with length 0


Could not resolve "KIC 5500000" to a sky position.
ERROR:lightkurve.search:Could not resolve "KIC 5500000" to a sky position.


❌ Error analysis for KIC 5500000: index 0 out of range for table with length 0


In [ ]:
def check_my_data(X, y):
    p_idx = np.where(y == 1)[0][0]
    n_idx = np.where(y == 0)[0][0]
    b_idx = np.where(y == 2)[0][0]

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].plot(X[p_idx], color='green')
    ax[0].set_title("Planet Signal (Target)")

    ax[1].plot(X[n_idx], color='gray')
    ax[1].set_title("Empty Signal (Noise)")

    ax[2].plot(X[b_idx], color='red')
    ax[2].set_title("Binary Signal (The Trap)")

    plt.show()


In [ ]:
stars_to_test = ["KIC 10848459", "KIC 9832227"] # تقدري تحطي أي أرقام هنا

for star in stars_to_test:
    try:
        # 1. سحب وتجهيز البيانات (بنفس الطريقة اللي اتعلمناها)
        # 2. الموديل يتوقع النتيجة
        prediction = multi_model.predict([input_g, input_l])
        final_choice = np.argmax(prediction[0])

        if final_choice == 1:
            print(f"🚨 ALERT! Potential Planet found in {star} with {prediction[0][1]*100:.2f}% confidence!")
        else:
            print(f"🌌 {star} seems to be just a normal star or binary.")
    except:
        continue

In [ ]:
multi_model.save("Exoplanet_Hunter_Model.h5")
print("💾 Project Saved! Your AI Brain is safe now.")

In [ ]:
# ملحوظة: دي نجوم ساطعة جداً بنستخدمها كمرجع للهدوء
stars_to_test = ["KIC 10848459", "KIC 9832227"]

for name in stars_to_test: # تم التصحيح هنا
    try:
        print(f"🛰️ Processing {name}...")
        search = lk.search_lightcurve(name, author="Kepler", cadence="long")
        if len(search) == 0: continue

        lc = search[0].download().remove_outliers().flatten()

        pg = lc.to_periodogram(method='bls')
        best_p = pg.period_at_max_power
        t0_p = pg.transit_time_at_max_power

        lc_folded = lc.fold(period=best_p, epoch_time=t0_p)
        bin_global = lc_folded.bin(time_bin_size=best_p/500)

        # تحويل لـ array صريح ومعالجة الـ Masked Array
        global_flux = np.array(bin_global.flux.value[:500])
        if hasattr(global_flux, 'filled'): # لو هو masked array
            global_flux = global_flux.filled(np.nanmean(global_flux))

        # ضبط الطول لـ 500
        if len(global_flux) < 500:
            global_flux = np.pad(global_flux, (0, 500 - len(global_flux)), mode='constant', constant_values=np.nanmean(global_flux))

        global_flux = np.nan_to_num(global_flux, nan=np.nanmean(global_flux))

        # الـ Local View باستخدام الفانكشن بتاعتك
        local_flux = get_local_view(global_flux, None, None)
        local_flux = np.nan_to_num(local_flux, nan=np.nanmean(local_flux))

        # الـ Scaling
        global_scaled = (global_flux - np.min(global_flux)) / (np.max(global_flux) - np.min(global_flux) + 1e-8)
        local_scaled = (local_flux - np.min(local_flux)) / (np.max(local_flux) - np.min(local_flux) + 1e-8)

        # الـ Prediction
        res = multi_model.predict([global_scaled.reshape(1, 500, 1),
                                   local_scaled.reshape(1, 100, 1)], verbose=0)

        predicted_class = np.argmax(res[0])
        confidence = res[0][predicted_class] * 100

        # رسم الخطوط اللي طلبكتيها (استدعاء الدالة)
        plot_with_lines(lc_folded, name, best_p)

        print(f"🧠 AI: {classes[predicted_class]} ({confidence:.2f}%)")

    except Exception as e:
        print(f"❌ Error with {name}: {e}")

In [ ]:
y_pred = multi_model.predict([X_g_final, X_l_final])
y_pred_classes = np.argmax(y_pred, axis=1)
cm = confusion_matrix(y_final, y_pred_classes)

# 3. الرسم الاحترافي
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Planet', 'Binary'],
            yticklabels=['Normal', 'Planet', 'Binary'])
plt.title('Scientific Verification: Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('AI Prediction')
plt.show()

In [ ]:
def final_strike(name):
    try:
        # 1. التحميل
        search = lk.search_lightcurve(name, author="Kepler", cadence="long")
        lc = search[0].download().remove_outliers().flatten()
        pg = lc.to_periodogram(method='bls')

        # 2. التجهيز
        lc_folded = lc.fold(period=pg.period_at_max_power, epoch_time=pg.transit_time_at_max_power)
        flux = lc_folded.bin(time_bin_size=pg.period_at_max_power/500).flux.value[:500]

        # 3. حل مشكلة الـ NaN (السر هنا)
        flux = np.nan_to_num(flux, nan=np.nanmean(flux))
        f_min, f_max = np.min(flux), np.max(flux)
        if f_max - f_min == 0: return "Dead Data" # لو البيانات ميتة

        scaled = (flux - f_min) / (f_max - f_min + 1e-8)

        # 4. التوقع والرسم في خبطة واحدة
        pred = multi_model.predict([scaled.reshape(1,500,1), get_local_view(scaled,None,None).reshape(1,100,1)], verbose=0)

        # ارسمي الرسمة اللي بتثبت نجاحك
        plot_with_lines(lc_folded, name, pg.period_at_max_power.value)

        return np.argmax(pred[0]), np.max(pred[0])
    except:
        return "Error", 0

# جربي النجم ده وقفلي بعدها:
print(final_strike("KIC 10848459"))